<a href="https://colab.research.google.com/github/sebastianatanasovici-wq/-notebook-/blob/main/pregunta_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from __future__ import annotations

from functools import wraps
from time import perf_counter


In [5]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper

# **2) Problema Charcha**

***Restricciones críticas:***

Balance de desechos

Tiempo total disponible

Límite de vertido al río

***Interpretación:***

El desecho es un recurso importante: se puede reutilizar (C y D) o botar

Si el límite del río se reduce → se produce más C y D

***Sensibilidad:***

Si aumenta el precio de C → se usará más desecho en C

Si aumenta el costo de D → se dejará de producir

Si aumenta el tiempo disponible → aumenta la utilidad

***Materia Prima***

Costo: 6 dólares por libra

**Se puede usar para producir:**

Insumo 1:

2 oz por libra

2 horas

Costo adicional: 2

Insumo 2:

3 oz por libra

2 horas

Costo adicional: 4

### **Procesos principales**

**Proceso 1**


Tiempo: 2 horas

Costo: 1

Produce:

1 A

1 desecho

**Proceso 2 **

Usa: 1 insumo 1 + 2 insumo 2

Tiempo: 3 horas

Costo: 8

Produce:

1 B

0.8 desecho

**Productos desde desechos**

Producto C

Usa: 2 insumo 1 + 0.8 desecho

Tiempo: 1 hora

Costo: 4

Venta: 11

Producto D

Usa: 2 insumo 2 + 1.2 desecho

Tiempo: 1 hora

Costo: 5

Venta: 7

**Restricciones**

Tiempo total ≤ 6000 horas

Desecho al río ≤ 1000 oz

Demanda:

A ≤ 5000

B ≤ 5000

C y D ilimitado



In [6]:
def create_ampl_instance(solver: str = 'highs'):
    return ampl_notebook(modules=[solver], license_uuid='default')


def extract_solution(ampl):
    return {
        "Insumo1_libras": round(ampl.get_variable("x1").value(), 2),
        "Insumo2_libras": round(ampl.get_variable("x2").value(), 2),
        "Proceso_A": round(ampl.get_variable("p1").value(), 2),
        "Proceso_B": round(ampl.get_variable("p2").value(), 2),
        "Producto_C": round(ampl.get_variable("c").value(), 2),
        "Producto_D": round(ampl.get_variable("d").value(), 2),
        "Desecho": round(ampl.get_variable("w").value(), 2),
        "Utilidad": round(ampl.get_objective("Utilidad").value(), 2),
    }


@timed
def solve_charcha_problem():
    ampl = create_ampl_instance()

    ampl.eval(r'''
        # VARIABLES
        var x1 >= 0;
        var x2 >= 0;
        var p1 >= 0;
        var p2 >= 0;
        var c >= 0;
        var d >= 0;
        var w >= 0;

        # FUNCION OBJETIVO
        maximize Utilidad:
            18*p1 + 24*p2 + 11*c + 7*d
            - (6*(x1+x2))
            - (2*x1 + 4*x2)
            - (1*p1 + 8*p2)
            - (4*c + 5*d);

        # RESTRICCIONES

        r1: 2*x1 >= 2*p1 + p2 + 2*c;
        r2: 3*x2 >= p1 + 2*p2 + 2*d;

        # balance de desechos
        r3: p1 + 0.8*p2 = 0.8*c + 1.2*d + w;

        # limite ambiental
        r4: w <= 1000;

        # tiempo disponible
        r5: 2*x1 + 2*x2 + 2*p1 + 3*p2 + c + d <= 6000;

        # demanda
        r6: p1 <= 5000;
        r7: p2 <= 5000;
    ''')

    ampl.option["solver"] = "highs"
    ampl.solve()

    return extract_solution(ampl)


resultado, tiempo = solve_charcha_problem()

print("=== RESULTADOS EJERCICIO 2 ===")
print(f"Solución -> {resultado}")
print(f"Tiempo   -> {tiempo:.6f} segundos")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 6366.336634
6 simplex iterations
0 barrier iterations
=== RESULTADOS EJERCICIO 2 ===
Solución -> {'Insumo1_libras': 1356.44, 'Insumo2_libras': 386.14, 'Proceso_A': 1158.42, 'Proceso_B': 0.0, 'Producto_C': 198.02, 'Producto_D': 0.0, 'Desecho': 1000.0, 'Utilidad': 6366.34}
Tiempo   -> 3.577170 segundos
